# 1.混合策略实现doc-doc对称检索

知识点讲解：HyDE 假设性文档嵌入与 doc-doc 对称检索

1. 什么是"非对称检索"及其固有缺陷
   - 常规 RAG 是 query-doc 非对称检索：用一个"短问句"的向量去匹配一堆"长文档"的向量
   - 两者在形态上严重不对称：
     * 长度不对称：问题 10~30 字，文档 300~500 字
     * 句式不对称：问题是疑问句，文档是陈述句
     * 词汇不对称：问题用口语词，文档用书面术语
   - Embedding 模型是在"文本对相似度"上训练的，形态差异本身就会拉大向量距离，
     即使语义完全相关，相似度也上不去——这是 RAG 效果的一个系统性瓶颈
   - 此外 query 与答案可能只存在弱相关性，导致难以找到最相关的文档

2. HyDE（Hypothetical Document Embeddings）的核心思想
   - 论文《Precise Zero-Shot Dense Retrieval without Relevance Labels》（arXiv:2212.10496）
   - 思路：既然 doc-doc 比 query-doc 更对称，那就先把 query 变成 doc！
   - 具体做法：让 LLM 针对用户问题"凭空写一篇假设性答案文档"
     （即使内容可能有事实错误也不要紧），然后用这篇假设文档的向量去检索真实文档
   - 于是检索变成了 doc-doc 对称检索：长度、句式、词汇分布都高度一致，向量匹配准确度显著提升

3. 为什么"事实可能错误的假设文档"依然有效
   - 检索阶段只关心"语义/主题/术语分布"的相似性，不关心事实正确性
   - LLM 生成的假设文档虽然细节可能编造，但会自然使用
     该领域的正确术语、表达方式和论述结构，把查询向量"拉近"到真实文档所在的向量空间区域
   - 关键认知：假设文档只是"检索的桥梁"，绝不会进入最终答案；
     最终答案仍基于检索出的真实文档生成，所以不存在幻觉扩散风险

4. HyDE Prompt 的设计要点（本例分析）
   - 本例使用 "请写一篇科学论文来回答这个问题" 这一表述
   - "科学论文"这个体裁提示至关重要：引导 LLM 输出长篇、书面化、术语密集的文本，
     形态上最接近技术文档
   - 若换成"请简短回答"，生成内容太短，就退化成 query-doc 非对称检索，效果打折
   - 生产环境应根据知识库文体调整体裁提示：技术文档→"技术文档/API 说明"；
     法律文本→"法律条文"；客服 FAQ→"标准答复"

5. 与 Step-Back 策略的实现对比
   - 两者的代码骨架几乎完全相同：BaseRetriever 子类 + {"question": Passthrough} | prompt | llm | parser | retriever
   - 唯一区别在 Prompt 的意图：
     * Step-Back：把问题"抽象化"，输出仍是一个问题（query → query）
     * HyDE：把问题"文档化"，输出是一段文章（query → doc）
   - 这体现了 LangChain 自定义检索器的可塑性：同一套骨架，只换 Prompt 就能实现不同检索策略

6. 链式管道的精妙之处
     {"question": RunnablePassthrough()} | prompt | self.llm | StrOutputParser() | self.retriever
   - 最后一环直接把 retriever 接入管道，因为它实现了 Runnable
     （输入 str、输出 List[Document]），正好承接 StrOutputParser 的输出
   - 一条管道同时完成"生成假设文档 + 用它检索"，代码极简

7. HyDE 的代价
   - 每次检索多一次 LLM 调用，且要生成较长文本（几百 token），延迟通常增加 2~5 秒，成本上升明显
   - 生成的假设文档很长，embed_query 的 token 消耗也随之增加
   - 因此 HyDE 更适合"离线批处理"或"对精度要求极高、能容忍延迟"的场景

8. 局限性与失败案例（来自课程文档 02 节）
   - 场景一：query 缺乏足够上下文时，HyDE 容易误解词语，产生错误假设文档导致检索失败
     例如问 "Bel 是什么？"，无上下文时 HyDE 把 Bel 误当成某人的化名，
     生成的假设文档完全偏题，从而检索不到相关文档
   - 场景二：开放式查询，HyDE 可能引入偏见
     例如 "作者会如何评价艺术与工程的区别？"，直接检索已能给出合理回答，
     经 HyDE 转换后反而生成带偏见的文档
   - 结论：HyDE 是不完全依赖 embedding、强调"答案与查找内容相似性"的无监督方法，
     能提升 RAG 效果，但若 LLM 无法理解用户问题，就不会产生最佳结果，需按场景决定是否采用

9. 最佳实践建议
   - 体裁提示要匹配知识库文体（论文/技术文档/法律条文/FAQ），这是效果的关键开关
   - 生成假设文档的 LLM 用 temperature=0，保证检索结果可复现；
     想增加召回多样性也可生成多篇假设文档再做 RRF 融合（HyDE + RAG-Fusion）
   - 明确认知：假设文档只用于检索，绝不能进入最终答案的上下文，否则会引入幻觉
   - 延迟敏感场景慎用；可对高频查询缓存假设文档，或降级为普通向量检索
   - 可与 Step-Back、Multi-Query 组合：先回退再 HyDE，或多个假设文档并行检索
   - 本例在 _get_relevant_documents 内部每次重建 Prompt 与链，属教学写法；
     生产环境应提前构建一次以复用
   - 建议内部调用时传 config={"callbacks": run_manager.get_child()}，保持追踪链完整
   - 假设文档过长可能超出 Embedding 模型输入上限（8191 token），可在 Prompt 中限定字数或截断


In [ ]:
from typing import List
import dotenv
import weaviate
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_weaviate import WeaviateVectorStore
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()

In [ ]:
class HyDERetriever(BaseRetriever):
    """HyDE混合策略检索器
    继承 BaseRetriever 实现 HyDE（假设性文档嵌入）策略：
    先让 LLM 针对问题生成一篇假设性答案文档，
    再用这篇文档去检索真实文档，从而把 query-doc 非对称检索
    转换为 doc-doc 对称检索，提升向量匹配精度。
    """
    retriever: BaseRetriever
    llm: BaseLanguageModel
    def _get_relevant_documents(
            self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        """传递检索query实现HyDE混合策略检索
        参数：
            query: str - 用户的原始问题（短问句形态）
            run_manager: CallbackManagerForRetrieverRun - 回调管理器（关键字参数）
        返回：
            List[Document] - 用假设性文档检索到的真实文档列表
        执行流程：
            1) 构建"生成假设性文档"的 Prompt
            2) 组装 "生成假设文档 → 用它检索" 的一体化链
            3) 执行链并直接返回真实文档列表
        """
        prompt = ChatPromptTemplate.from_template(
            "请写一篇科学论文来回答这个问题。\n"
            "问题: {question}\n"
            "文章: "
        )
        chain = (
                {"question": RunnablePassthrough()}
                | prompt
                | self.llm
                | StrOutputParser()
                | self.retriever
        )
        return chain.invoke(query)

In [ ]:
db = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://mbakeruerziae6psyex7ng.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("ZltPVa9ZSOxUcfafelsggGyyH6tnTYQYJvBx"),
    ),
    index_name="DatasetDemo",
    text_key="text",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
retriever = db.as_retriever(search_type="mmr")
hyde_retriever = HyDERetriever(
    retriever=retriever,
    llm=ChatOpenAI(model="deepseek-v4-pro", temperature=0),
)
documents = hyde_retriever.invoke("关于LLMOps应用配置的文档有哪些？")
print(documents)
print(len(documents))